# 22 — Supervised Fine-Tuning from First Principles

## Goal

Understand how a pretrained autoregressive language model
learns to follow instructions through supervised examples.

We focus on:

- Conversation formatting and role boundaries.
- Teacher-forced next-token prediction.
- Shifted labels and assistant-only loss masking.
- Fine-tuning a pretrained GPT.
- Evaluating instruction-following behavior.

## 01. SFT Objective and Label Alignment

The model still predicts the next token.

The difference is which tokens provide supervision.

For assistant-only SFT, the prompt remains part of the
conditioning context, but only selected assistant targets
contribute to the training loss.

In [1]:
import torch
import torch.nn.functional as F
from llmfp.models import GPT, GPTConfig

In [2]:
# Align next-token targets with an assistant-only loss mask.

# Vocabulary:
# 0 = BOS
# 1 = USER
# 2 = QUESTION
# 3 = ASSISTANT
# 4 = ANSWER
# 5 = EOS

# (B=1, L=6): sequences, full conversation token positions
token_ids = torch.tensor([[0, 1, 2, 3, 4, 5]])

# (B=1, T=5): context tokens supplied to the language model
input_ids = token_ids[:, :-1]

# (B=1, T=5): next-token labels for each input position
target_ids = token_ids[:, 1:]

# (B=1, T=5): which target positions contribute to SFT loss
assistant_mask = torch.tensor([[False, False, False, True, True]])

assert input_ids.shape == target_ids.shape == assistant_mask.shape
assert torch.equal(target_ids[assistant_mask], torch.tensor([4, 5]))

print("Inputs:", input_ids.tolist())
print("Targets:", target_ids.tolist())
print("Loss mask:", assistant_mask.tolist())

Inputs: [[0, 1, 2, 3, 4]]
Targets: [[1, 2, 3, 4, 5]]
Loss mask: [[False, False, False, True, True]]


In [3]:
# Verify that only assistant targets contribute gradients.

vocab_size = 6

# (B=1, T=5, V=6): next-token logits for every prediction position
logits = torch.zeros((1, 5, vocab_size), requires_grad=True)

# (B=1, T=5): unreduced next-token negative log-likelihoods
per_token_loss = F.cross_entropy(
    logits.reshape(-1, vocab_size), target_ids.reshape(-1), reduction="none"
).reshape(1, 5)

# (): mean NLL over assistant target positions only
sft_loss = per_token_loss[assistant_mask].mean()

sft_loss.backward()

assert logits.grad is not None

# Prompt prediction positions must contribute zero gradients.
assert torch.all(logits.grad[:, :3, :] == 0)

# Assistant prediction positions must receive gradients.
assert torch.any(logits.grad[:, 3:, :] != 0)

print("Masked SFT loss:", sft_loss.item())
print("Assistant-only gradient test passed.")

Masked SFT loss: 1.7917594909667969
Assistant-only gradient test passed.


## 02. Instruction Data and Chat Templates

A chat template converts structured role/content messages into
a token sequence understood by the language model.

For our first experiment, we define a minimal single-turn format:

[BOS, USER, question..., ASSISTANT, answer..., EOS]

The prompt and role markers provide context. We select the answer
tokens and EOS as supervised prediction targets.

This is a pedagogical symbolic vocabulary, not a pretrained
model tokenizer.

In [4]:
BOS_ID = 0
USER_ID = 1
ASSISTANT_ID = 3
EOS_ID = 5


def encode_sft_pair(
    question: list[int],
    answer: list[int],
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Serialize one instruction-response pair for assistant-only SFT.

    Args:
        question: Token IDs of the user content.
        answer: Token IDs of the assistant response.

    Returns:
        input_ids: (T,) — context tokens.
        target_ids: (T,) — next-token labels.
        loss_mask: (T,) — True for answer and EOS targets.
    """
    if not question or not answer:
        raise ValueError("Question and answer must be non-empty.")

    # Serialize the full conversation, including role delimiters.
    full_tokens = [BOS_ID, USER_ID, *question, ASSISTANT_ID, *answer, EOS_ID]

    # (L,): complete conversation token positions
    sequence = torch.tensor(full_tokens, dtype=torch.long)

    # Position of the first assistant answer token in the full sequence
    answer_start = 3 + len(question)

    # (T=L-1,): input tokens for next-token prediction
    input_ids = sequence[:-1]

    # (T=L-1,): next-token targets, shifted by one position
    target_ids = sequence[1:]

    # (T=L-1,): supervised target positions
    loss_mask = torch.zeros_like(target_ids, dtype=torch.bool)

    # Target index is one position earlier than its full-sequence index.
    # Include answer tokens and EOS, but exclude the assistant role marker.
    loss_mask[answer_start - 1 :] = True

    return input_ids, target_ids, loss_mask

In [5]:
# Two user-content tokens and two assistant-content tokens
question = [2, 6]
answer = [4, 7]

input_ids, target_ids, loss_mask = encode_sft_pair(question, answer)

print("Inputs: ", input_ids.tolist())
print("Targets:", target_ids.tolist())
print("Mask:   ", loss_mask.tolist())

# Only answer tokens and EOS receive supervised loss.
torch.testing.assert_close(
    target_ids[loss_mask],
    torch.tensor([4, 7, 5]),
)

# Two answer tokens plus one end-of-sequence token
assert loss_mask.sum().item() == 3

print("SFT encoding test passed.")

Inputs:  [0, 1, 2, 6, 3, 4, 7]
Targets: [1, 2, 6, 3, 4, 7, 5]
Mask:    [False, False, False, False, True, True, True]
SFT encoding test passed.


### Chat Templates and SFT Encoding — Takeaways

A chat template maps structured messages into a deterministic
token sequence containing role delimiters and content tokens.

For a single-turn assistant-only SFT example, we constructed:

- Shifted model inputs.
- Next-token prediction targets.
- A Boolean mask selecting assistant answer tokens and EOS.

The mask is aligned with target positions, not the original
unshifted conversation indices.

Causal attention masks and supervision masks solve different
problems: one controls accessible information; the other controls
which prediction errors contribute to optimization.

The encoding is intentionally minimal. Multi-turn conversations,
real tokenizers, padding, and truncation require additional care.

In [ ]:
torch.manual_seed(42)

sft_config = GPTConfig(
    vocab_size=8,
    embedding_dim=32,
    num_query_heads=4,
    num_kv_heads=2,
    num_layers=2,
    hidden_dim=64,
)

sft_model = GPT(sft_config)
sft_model.train()

# Each tensor has shape (T=7,): shifted token positions.
input_ids, target_ids, loss_mask = encode_sft_pair(
    question=[2, 6], answer=[4, 7]
)

# (B=1, T=7): sequences, input-token positions
sft_inputs = input_ids.unsqueeze(0)

# (B=1, T=7): sequences, next-token targets
sft_targets = target_ids.unsqueeze(0)

# (B=1, T=7): supervised assistant-target positions
sft_mask = loss_mask.unsqueeze(0)

assert sft_inputs.shape == sft_targets.shape == sft_mask.shape
assert sft_mask.sum().item() == 3

print("Inputs:", sft_inputs.tolist())
print("Targets:", sft_targets.tolist())
print("Mask:", sft_mask.tolist())

Inputs: [[0, 1, 2, 6, 3, 4, 7]]
Targets: [[1, 2, 6, 3, 4, 7, 5]]
Mask: [[False, False, False, False, True, True, True]]


In [ ]:
# (B=1, T=7, V=8): next-token logits at every sequence position
sft_logits = sft_model(sft_inputs)

assert isinstance(sft_logits, torch.Tensor)

# (B=1, T=7): unreduced next-token negative log-likelihoods
per_token_loss = F.cross_entropy(
    sft_logits.reshape(-1, sft_config.vocab_size),
    sft_targets.reshape(-1),
    reduction="none",  # !not "mean"!
).reshape(sft_targets.shape)

# (): mean NLL over assistant targets only
sft_loss = per_token_loss[sft_mask].mean()

# (): ordinary next-token NLL over every position, for comparison
full_lm_loss = per_token_loss.mean()

print("Full LM loss:", full_lm_loss.item())
print("Assistant-only SFT loss:", sft_loss.item())

assert sft_loss.requires_grad
assert torch.isfinite(sft_loss)

Full LM loss: 2.136887311935425
Assistant-only SFT loss: 2.085982084274292


In [8]:
# Keep the graph because we will also backpropagate to model parameters.
# (B=1, T=7, V=8): gradients of SFT loss w.r.t. vocabulary logits
(logits_grad,) = torch.autograd.grad(
    outputs=sft_loss, inputs=(sft_logits,), retain_graph=True
)

# (N_prompt=4, V=8): gradients at masked predictor positions
prompt_logit_grads = logits_grad[~sft_mask]

# (N_assistant=3, V=8): gradients at supervised predictor positions
assistant_logit_grads = logits_grad[sft_mask]

assert torch.all(prompt_logit_grads == 0)
assert torch.any(assistant_logit_grads != 0)

print("Prompt logits gradient norm:", prompt_logit_grads.norm().item())

print("Assistant logits gradient norm:", assistant_logit_grads.norm().item())

print("Direct logit gradient masking passed.")

Prompt logits gradient norm: 0.0
Assistant logits gradient norm: 0.5508262515068054
Direct logit gradient masking passed.


In [9]:
sft_model.zero_grad(set_to_none=True)

sft_loss.backward()

# (V=8, C=32): gradients of the trainable token embedding table
embedding_grads = sft_model.token_embedding.weight.grad

assert embedding_grads is not None

# (N_prompt=5,): IDs appearing only in the prompt/context
prompt_token_ids = torch.tensor([BOS_ID, USER_ID, 2, 6, ASSISTANT_ID])

# (N_prompt=5, C=32): embedding gradients for prompt-only tokens
prompt_embedding_grads = embedding_grads[prompt_token_ids]

# (N_prompt=5,): gradient norm of each prompt token embedding
prompt_embedding_norms = prompt_embedding_grads.norm(dim=-1)

print("Prompt token IDs:", prompt_token_ids.tolist())
print("Prompt embedding gradient norms:", prompt_embedding_norms.tolist())

assert torch.any(prompt_embedding_norms > 0)

print("Indirect prompt gradient flow passed.")

Prompt token IDs: [0, 1, 2, 6, 3]
Prompt embedding gradient norms: [0.03976085036993027, 0.03468206897377968, 0.02956317365169525, 0.03257538005709648, 0.2189846783876419]
Indirect prompt gradient flow passed.


In [11]:
# (C=32,): EOS input-embedding gradient
eos_embedding_grad = embedding_grads[EOS_ID]

# (V=8, C=32): gradients of the vocabulary output projection
lm_head_grads = sft_model.lm_head.weight.grad

assert lm_head_grads is not None

# (C=32,): EOS output-class projection gradient
eos_output_grad = lm_head_grads[EOS_ID]

print("EOS embedding gradient:", eos_embedding_grad.norm().item())
print("EOS LM-head gradient:", eos_output_grad.norm().item())

assert torch.all(eos_embedding_grad == 0)
assert eos_output_grad.norm().item() > 0

print("Input embedding / output head gradient distinction passed.")

EOS embedding gradient: 0.0
EOS LM-head gradient: 1.5281026363372803
Input embedding / output head gradient distinction passed.


### Assistant-only Loss and Gradient Flow — Takeaways

We implemented assistant-only SFT using a real causal GPT.

The objective selects assistant response targets from
teacher-forced next-token predictions.

We verified two distinct gradient properties:

1. Predictor logits at masked prompt positions receive no
   direct gradient from the SFT loss.
2. Prompt token embeddings can still receive gradients because
   supervised assistant predictions depend on earlier context
   through causal attention.

Loss masking controls which predictions are supervised.
It does not remove the prompt from the Transformer's context.

We also distinguished input embeddings from vocabulary output
projections using the EOS token.

Our GPT uses separate embedding and LM-head parameters; the
behavior would differ under weight tying.

The next step is to combine examples of different lengths
using padding and valid assistant-target masks.


## 04. Padding, Batching, and Loss Masks

SFT datasets contain instruction-response examples of different lengths.

To combine them into a dense batch, we right-pad the shifted sequences
to a common length.

We distinguish:

- `valid_mask`: positions belonging to the original example.
- `loss_mask`: positions whose next-token targets are supervised.

For assistant-only SFT, the loss mask selects answer tokens and EOS.
Padding positions must not contribute to the loss.


In [12]:
# Reuse encode_sft_pair from Section 02.
examples = [
    encode_sft_pair(question=[2], answer=[4]),
    encode_sft_pair(question=[2, 6], answer=[4, 7]),
]

batch_size = len(examples)
max_length = max(x.numel() for x, _, _ in examples)

# (B=2, T_max=7): sequences, padded predictor inputs
batch_inputs = torch.full(
    (batch_size, max_length), fill_value=BOS_ID, dtype=torch.long
)

# (B=2, T_max=7): sequences, padded next-token targets
batch_targets = torch.full_like(batch_inputs, BOS_ID)

# (B=2, T_max=7): positions belonging to original examples
batch_valid_mask = torch.zeros_like(batch_inputs, dtype=torch.bool)

# (B=2, T_max=7): selected assistant-target positions
batch_loss_mask = torch.zeros_like(batch_inputs, dtype=torch.bool)

for row, (inputs, targets, mask) in enumerate(examples):
    length = inputs.numel()

    batch_inputs[row, :length] = inputs
    batch_targets[row, :length] = targets
    batch_valid_mask[row, :length] = True
    batch_loss_mask[row, :length] = mask

print("Inputs:", batch_inputs.tolist())
print("Targets:", batch_targets.tolist())
print("Valid mask:", batch_valid_mask.tolist())
print("Loss mask:", batch_loss_mask.tolist())

assert batch_inputs.shape == (2, 7)
assert batch_loss_mask.sum().item() == 5
assert not batch_loss_mask[~batch_valid_mask].any().item()
assert batch_inputs.max().item() < sft_config.vocab_size


Inputs: [[0, 1, 2, 3, 4, 0, 0], [0, 1, 2, 6, 3, 4, 7]]
Targets: [[1, 2, 3, 4, 5, 0, 0], [1, 2, 6, 3, 4, 7, 5]]
Valid mask: [[True, True, True, True, True, False, False], [True, True, True, True, True, True, True]]
Loss mask: [[False, False, False, True, True, False, False], [False, False, False, False, True, True, True]]


In [13]:
# (B=2, T=7, V=8): next-token vocabulary logits
batch_logits = sft_model(batch_inputs)

# (B=2, T=7): unreduced NLL for every target position
batch_nll = F.cross_entropy(
    batch_logits.reshape(-1, sft_config.vocab_size),
    batch_targets.reshape(-1),
    reduction="none",
).reshape(batch_targets.shape)

# Prevent an empty supervised set from producing NaN.
if not batch_loss_mask.any().item():
    raise ValueError("No supervised targets in the batch.")

# (): Boolean-mask mean over supervised tokens
boolean_loss = batch_nll[batch_loss_mask].mean()

# (B=2, T=7): -100 marks targets excluded from CE
ignored_targets = batch_targets.masked_fill(~batch_loss_mask, -100)

# (): Cross-entropy ignores -100 targets in its mean
ignore_index_loss = F.cross_entropy(
    batch_logits.reshape(-1, sft_config.vocab_size),
    ignored_targets.reshape(-1),
    ignore_index=-100,
)

torch.testing.assert_close(boolean_loss, ignore_index_loss)

print("Boolean-mask loss:", boolean_loss.item())
print("Ignore-index loss:", ignore_index_loss.item())
print("Supervised tokens:", batch_loss_mask.sum().item())


Boolean-mask loss: 1.987636923789978
Ignore-index loss: 1.987636923789978
Supervised tokens: 5


In [14]:
# (B=2, T=7, V=8): logit gradients using Boolean masking
(grad_boolean,) = torch.autograd.grad(
    boolean_loss, batch_logits, retain_graph=True
)

# (B=2, T=7, V=8): logit gradients using ignore_index
(grad_ignored,) = torch.autograd.grad(ignore_index_loss, batch_logits)

torch.testing.assert_close(grad_boolean, grad_ignored)

# Unsupervised positions must have zero direct logit gradients.
assert torch.all(grad_boolean[~batch_loss_mask] == 0)

print("Both masking methods produce equivalent gradients.")


Both masking methods produce equivalent gradients.



### Padding, Batching, and Loss Masks — Takeaways

We constructed a batch containing variable-length SFT examples.

Right padding aligns their tensor shapes while preserving their
original token positions.

The valid mask identifies original sequence positions, whereas
the loss mask selects assistant response targets and EOS.

Boolean masking and `ignore_index=-100` produce equivalent
cross-entropy losses and gradients when they select the same
targets and use the same normalization.

Loss averaging is a modeling choice: token-average and
example-average objectives give different weights to examples
with different response lengths.

The next step is to fine-tune a pretrained GPT using a compatible
tokenizer and instruction dataset.



## 05. Fine-Tuning a Pretrained GPT

### 05.1 Restoring the Pretrained Model and Tokenizer

Fine-tuning starts from pretrained parameters rather than a random
initialization.

We reuse the Tiny Shakespeare checkpoint and reconstruct the
character-level tokenizer from the same training corpus.

The tokenizer vocabulary and the model's input/output dimensions
must remain compatible.


In [15]:
from pathlib import Path
from llmfp.data import CharacterTokenizer

# Reuse the data and best-checkpoint paths from Lesson 16.
corpus_path = Path("../data/raw/tinyshakespeare.txt")
checkpoint_path = Path("../checkpoints/tiny_shakespeare/step_02000.pt")

for path in (corpus_path, checkpoint_path):
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing {path}; working directory: {Path.cwd()}"
        )

corpus_text = corpus_path.read_text(encoding="utf-8")

# Reconstruct the deterministic character vocabulary.
pretrain_tokenizer = CharacterTokenizer.from_text(corpus_text)

assert pretrain_tokenizer.vocab_size == 65

print("Vocabulary size:", pretrain_tokenizer.vocab_size)
print("Checkpoint:", checkpoint_path.name)


Vocabulary size: 65
Checkpoint: step_02000.pt


In [16]:
# Lesson 16 saved a GPTConfig dataclass in the checkpoint.
with torch.serialization.safe_globals([GPTConfig]):
    pretrained_checkpoint = torch.load(
        checkpoint_path, map_location="cpu", weights_only=True
    )

pretrained_config = pretrained_checkpoint["model_config"]

assert isinstance(pretrained_config, GPTConfig)
assert pretrained_config.vocab_size == pretrain_tokenizer.vocab_size

# Restore the original architecture before loading the weights.
pretrained_gpt = GPT(pretrained_config)

pretrained_gpt.load_state_dict(
    pretrained_checkpoint["model_state_dict"],
    strict=True,
)

pretrained_gpt.eval()

print("Pretraining step:", pretrained_checkpoint["step"])
print("Vocabulary size:", pretrained_config.vocab_size)
print("Parameter count:", sum(p.numel() for p in pretrained_gpt.parameters()))


Pretraining step: 2000
Vocabulary size: 65
Parameter count: 3050240


In [17]:
prompt_text = "User: Say hello\nAssistant: "
answer_text = "Hello\n"

full_text = prompt_text + answer_text

prompt_tokens = pretrain_tokenizer.encode(prompt_text)
answer_tokens = pretrain_tokenizer.encode(answer_text)
full_tokens = pretrain_tokenizer.encode(full_text)

# (L,): complete character-token conversation
sequence = torch.tensor(full_tokens, dtype=torch.long)

# (B=1, T=L-1): shifted context tokens
pretrain_sft_inputs = sequence[:-1].unsqueeze(0)

# (B=1, T=L-1): shifted next-character targets
pretrain_sft_targets = sequence[1:].unsqueeze(0)

# (B=1, T=L-1): selected assistant-response target positions
pretrain_sft_mask = torch.zeros_like(pretrain_sft_targets, dtype=torch.bool)

# The first response character is predicted from the last prompt token.
answer_start = len(prompt_tokens) - 1
pretrain_sft_mask[:, answer_start:] = True

# (N_response,): answer characters and final newline targets
selected_targets = pretrain_sft_targets[pretrain_sft_mask]

torch.testing.assert_close(
    selected_targets,
    torch.tensor(answer_tokens),
)

print("Full text:", repr(full_text))
print(
    "Supervised text:",
    repr(pretrain_tokenizer.decode(selected_targets.tolist())),
)


Full text: 'User: Say hello\nAssistant: Hello\n'
Supervised text: 'Hello\n'


In [18]:
with torch.no_grad():
    # (B=1, T, V=65): pretrained model predictions
    baseline_logits = pretrained_gpt(pretrain_sft_inputs)

    # (B=1, T): negative log-likelihood for each target
    baseline_nll = F.cross_entropy(
        baseline_logits.reshape(-1, pretrained_config.vocab_size),
        pretrain_sft_targets.reshape(-1),
        reduction="none",
    ).reshape(pretrain_sft_targets.shape)

    # (): mean NLL over response characters and final newline
    baseline_sft_loss = baseline_nll[pretrain_sft_mask].mean()

print("Pre-fine-tuning SFT loss:", baseline_sft_loss.item())

assert torch.isfinite(baseline_sft_loss)


Pre-fine-tuning SFT loss: 3.8167755603790283



### Pretrained Model and Tokenizer — Takeaways

Fine-tuning begins from a pretrained checkpoint rather than
randomly initialized model parameters.

The tokenizer must preserve the original token-to-ID mapping.

Our Tiny Shakespeare GPT uses a character-level vocabulary with
65 tokens and has no dedicated chat-role or EOS tokens.

We serialized an instruction-response pair using ordinary
characters and aligned the assistant-only loss mask with
shifted targets.

We recorded a pre-fine-tuning SFT loss as a baseline.
No model parameters were updated in this step.
